# EXP12: 年度稳定性 + 蒙特卡洛 + 统计检验

1. 逐年 PnL/Sharpe/MaxDD/WR 分解
2. 蒙特卡洛模拟（打乱交易顺序）置信区间
3. PSR/DSR 统计检验
4. 滚动 Sharpe 稳定性

In [ ]:
import sys; sys.path.insert(0, '..')
from backtest import DataBundle, run_variant, C12_KWARGS
from backtest.stats import calc_stats
import pandas as pd
import numpy as np

data = DataBundle.load_default()

LIVE_KWARGS = {
    **C12_KWARGS,
    "intraday_adaptive": True,
    "choppy_threshold": 0.35,
    "kc_only_threshold": 0.60,
    "spread_cost": 0.50,
}

baseline = run_variant(data, "Baseline", **LIVE_KWARGS)
trades = baseline['_trades']
print(f"Baseline: {len(trades)} trades, PnL=${sum(t.pnl for t in trades):.0f}, Sharpe={baseline['sharpe']:.2f}")

## Part 1: 逐年分解

In [ ]:
df = pd.DataFrame([{
    'year': t.entry_time.year,
    'pnl': t.pnl,
    'strategy': t.strategy,
    'direction': t.direction,
    'win': 1 if t.pnl > 0 else 0,
} for t in trades])

yearly = df.groupby('year').agg(
    count=('pnl', 'count'),
    total_pnl=('pnl', 'sum'),
    avg_pnl=('pnl', 'mean'),
    win_rate=('win', 'mean'),
    std_pnl=('pnl', 'std'),
).round(2)

yearly['sharpe_approx'] = (yearly['avg_pnl'] / yearly['std_pnl'] * np.sqrt(252)).round(2)

print("=== Yearly Breakdown ===")
print(yearly)

pos_years = (yearly['total_pnl'] > 0).sum()
total_years = len(yearly)
print(f"\nPositive years: {pos_years}/{total_years} ({pos_years/total_years*100:.0f}%)")

In [ ]:
# 逐年 MaxDD
print("\n=== Yearly MaxDD ===")
for year in sorted(df['year'].unique()):
    year_trades = [t for t in trades if t.entry_time.year == year]
    equity = np.cumsum([t.pnl for t in year_trades])
    peak = np.maximum.accumulate(equity)
    dd = equity - peak
    max_dd = dd.min() if len(dd) > 0 else 0
    print(f"  {year}: {len(year_trades)} trades, PnL=${sum(t.pnl for t in year_trades):8.0f}, MaxDD=${max_dd:8.0f}")

## Part 2: 蒙特卡洛模拟

In [ ]:
pnls = np.array([t.pnl for t in trades])
n_sims = 5000
np.random.seed(42)

mc_total_pnls = []
mc_max_dds = []
mc_sharpes = []

for _ in range(n_sims):
    shuffled = np.random.permutation(pnls)
    mc_total_pnls.append(shuffled.sum())
    equity = np.cumsum(shuffled)
    peak = np.maximum.accumulate(equity)
    mc_max_dds.append((equity - peak).min())
    if shuffled.std() > 0:
        mc_sharpes.append(shuffled.mean() / shuffled.std() * np.sqrt(252))

mc_total_pnls = np.array(mc_total_pnls)
mc_max_dds = np.array(mc_max_dds)
mc_sharpes = np.array(mc_sharpes)

print(f"=== Monte Carlo ({n_sims} simulations) ===")
print(f"Total PnL: mean=${mc_total_pnls.mean():.0f}, "
      f"5%=${np.percentile(mc_total_pnls, 5):.0f}, "
      f"50%=${np.percentile(mc_total_pnls, 50):.0f}, "
      f"95%=${np.percentile(mc_total_pnls, 95):.0f}")
print(f"MaxDD: mean=${mc_max_dds.mean():.0f}, "
      f"5%=${np.percentile(mc_max_dds, 5):.0f}, "
      f"50%=${np.percentile(mc_max_dds, 50):.0f}, "
      f"95%=${np.percentile(mc_max_dds, 95):.0f}")
print(f"Sharpe: mean={mc_sharpes.mean():.2f}, "
      f"5%={np.percentile(mc_sharpes, 5):.2f}, "
      f"95%={np.percentile(mc_sharpes, 95):.2f}")

# 破产概率
ruin_pct = (mc_max_dds < -1500).mean() * 100
print(f"\nRuin prob (MaxDD > $1500): {ruin_pct:.1f}%")
ruin_3k = (mc_max_dds < -3000).mean() * 100
print(f"Ruin prob (MaxDD > $3000): {ruin_3k:.1f}%")

## Part 3: 滚动 Sharpe

In [ ]:
# 滚动 100 笔 Sharpe
window = 100
rolling_sharpes = []
for i in range(window, len(pnls)):
    chunk = pnls[i-window:i]
    if chunk.std() > 0:
        s = chunk.mean() / chunk.std() * np.sqrt(252)
        rolling_sharpes.append({'trade_idx': i, 'sharpe': s})

rs_df = pd.DataFrame(rolling_sharpes)
print(f"=== Rolling {window}-trade Sharpe ===")
print(f"Mean: {rs_df['sharpe'].mean():.2f}")
print(f"Std: {rs_df['sharpe'].std():.2f}")
print(f"Min: {rs_df['sharpe'].min():.2f}")
print(f"Max: {rs_df['sharpe'].max():.2f}")
print(f"% of windows with Sharpe > 0: {(rs_df['sharpe'] > 0).mean()*100:.1f}%")
print(f"% of windows with Sharpe > 0.5: {(rs_df['sharpe'] > 0.5).mean()*100:.1f}%")

## Part 4: 按策略逐年

In [ ]:
print("=== Yearly PnL by Strategy ===")
pivot = df.pivot_table(index='year', columns='strategy', values='pnl', aggfunc='sum')
print(pivot.round(0))

print("\n=== Yearly Trade Count by Strategy ===")
pivot_n = df.pivot_table(index='year', columns='strategy', values='pnl', aggfunc='count')
print(pivot_n)